<a href="https://colab.research.google.com/github/Haritha0105/Statistical-Learning-e21172/blob/main/Assignment_4_Data_Wrangling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.stats as stats

class PlottingMethods:
    """
    Modular plotting class that returns HTML-wrapped Plotly figures
    for flexible web embedding.
    """
    @staticmethod
    def generate_bar_chart(df, column, title="Bar Chart"):
        if df is None or column not in df.columns:
            return "<p>Data or Column not found.</p>"
        counts = df[column].value_counts().reset_index()
        counts.columns = [column, 'Count']
        counts['Percentage'] = (counts['Count'] / counts['Count'].sum()) * 100
        fig = px.bar(counts, x=column, y='Count', text=counts['Percentage'].apply(lambda x: f'{x:.1f}%'),
                     title=title, labels={'Count': 'Raw Counts'})
        fig.update_traces(textposition='outside')
        return fig.to_html(full_html=False)

    @staticmethod
    def generate_pie_chart(df, column, title="Pie Chart"):
        if df is None or column not in df.columns:
            return "<p>Data or Column not found.</p>"
        fig = px.pie(df, names=column, title=title)
        return fig.to_html(full_html=False)

    @staticmethod
    def generate_histogram(df, column, title="Histogram"):
        if df is None or column not in df.columns:
            return "<p>Data or Column not found.</p>"
        fig = px.histogram(df, x=column, title=title, marginal="box")
        return fig.to_html(full_html=False)

In [8]:
class DataInspector:
    """
    An end-to-end tool for CSV data ingestion, advanced cleaning,
    feature engineering preparation, and high-level statistical visualization.
    """
    def __init__(self):
        self.df = None

    def upload_data(self):
        try:
            from google.colab import files
            uploaded = files.upload()
            if not uploaded:
                print("No file uploaded.")
                return None
            file_name = list(uploaded.keys())[0]
            self.df = pd.read_csv(file_name, na_values=['?', 'n/a', 'NULL', ' ', ''])
            print(f"Successfully loaded {file_name}")
            self.auto_type_correction()
            return self.df
        except Exception as e:
            print(f"Error uploading data: {e}")
            return None

    def auto_type_correction(self):
        if self.df is None: return
        for col in self.df.columns:
            if self.df[col].dtype == 'object':
                converted = pd.to_numeric(self.df[col], errors='coerce')
                if not converted.isna().all():
                    self.df[col] = converted
                    print(f"Column '{col}' force-converted to Numeric.")

    def data_summary(self):
        if self.df is None:
            print("No data loaded.")
            return
        print("--- DATASET SUMMARY ---")
        print(f"Rows: {self.df.shape[0]} | Columns: {self.df.shape[1]}\n")
        num_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        cat_cols = self.df.select_dtypes(exclude=[np.number]).columns.tolist()
        print(f"Numerical Columns ({len(num_cols)}): {num_cols}")
        print(f"Categorical Columns ({len(cat_cols)}): {cat_cols}\n")
        print("--- FIRST 20 ROWS PREVIEW ---")
        display(self.df.head(20))

    def handle_missing_values(self, strategy='mean', constant_value=None):
        if self.df is None: return
        for col in self.df.columns:
            if self.df[col].isna().sum() > 0:
                if self.df[col].dtype in [np.number]:
                    if strategy == 'mean': self.df[col] = self.df[col].fillna(self.df[col].mean())
                    elif strategy == 'median': self.df[col] = self.df[col].fillna(self.df[col].median())
                    elif strategy == 'mode': self.df[col] = self.df[col].fillna(self.df[col].mode()[0])
                    elif strategy == 'constant' and constant_value is not None: self.df[col] = self.df[col].fillna(constant_value)
                else:
                    if strategy in ['mean', 'median', 'mode']: self.df[col] = self.df[col].fillna(self.df[col].mode()[0])
                    elif strategy == 'constant' and constant_value is not None: self.df[col] = self.df[col].fillna(constant_value)
        print(f"Missing values imputed using '{strategy}' strategy.")

    def remove_duplicates(self):
        if self.df is None: return
        initial_rows = self.df.shape[0]
        self.df.drop_duplicates(inplace=True)
        print(f"Removed {initial_rows - self.df.shape[0]} duplicate rows.")

    def handle_outliers(self, columns, action='delete'):
        if self.df is None: return
        for col in columns:
            if col in self.df.columns and self.df[col].dtype in [np.number]:
                Q1 = self.df[col].quantile(0.25)
                Q3 = self.df[col].quantile(0.75)
                IQR = Q3 - Q1
                lower_bound = Q1 - 1.5 * IQR
                upper_bound = Q3 + 1.5 * IQR
                outliers_mask = (self.df[col] < lower_bound) | (self.df[col] > upper_bound)
                if action == 'delete':
                    self.df = self.df[~outliers_mask]
                    print(f"Deleted outliers from column '{col}'.")
                elif action == 'flag':
                    self.df[f'{col}_outlier'] = outliers_mask.astype(int)
                    print(f"Flagged outliers for column '{col}'.")

    def delete_rows(self):
        if self.df is None: return
        user_input = input("Enter comma-separated row indices to delete (e.g., 0, 5, 22): ")
        if user_input.strip():
            indices = [int(x.strip()) for x in user_input.split(',') if x.strip().isdigit()]
            self.df.drop(index=indices, errors='ignore', inplace=True)
            print(f"Dropped rows: {indices}")

    def delete_columns(self):
        if self.df is None: return
        user_input = input("Enter comma-separated column names to delete: ")
        if user_input.strip():
            cols = [x.strip() for x in user_input.split(',')]
            self.df.drop(columns=cols, errors='ignore', inplace=True)
            print(f"Dropped columns: {cols}")

    def extract_normalized_numeric_data(self, strategy='standard'):
        numeric_df = self.df.select_dtypes(include=[np.number]).copy()
        numeric_df = numeric_df.loc[:, ~numeric_df.columns.str.endswith('_outlier')]
        if strategy == 'minmax': return (numeric_df - numeric_df.min()) / (numeric_df.max() - numeric_df.min())
        elif strategy == 'standard': return (numeric_df - numeric_df.mean()) / numeric_df.std()
        elif strategy == 'robust': return (numeric_df - numeric_df.median()) / (numeric_df.quantile(0.75) - numeric_df.quantile(0.25))
        return numeric_df

    def extract_normalized_categorical_data(self, strategy='onehot'):
        cat_df = self.df.select_dtypes(exclude=[np.number]).copy()
        if cat_df.empty: return pd.DataFrame()
        if strategy == 'onehot': return pd.get_dummies(cat_df, drop_first=True, dtype=float)
        elif strategy == 'ordinal' or strategy == 'uniform':
            encoded_df = pd.DataFrame()
            for col in cat_df.columns:
                labels, uniques = pd.factorize(cat_df[col])
                if strategy == 'uniform' and len(uniques) > 1: encoded_df[col] = labels / (len(uniques) - 1)
                else: encoded_df[col] = labels
            return encoded_df
        return cat_df

    def merge_datasets(self, num_strategy='standard', cat_strategy='onehot'):
        num_scaled = self.extract_normalized_numeric_data(strategy=num_strategy)
        cat_encoded = self.extract_normalized_categorical_data(strategy=cat_strategy)
        return pd.concat([num_scaled, cat_encoded], axis=1)

    def plot_univariate_subplots(self, column):
        if self.df is None or column not in self.df.columns: return
        fig = make_subplots(rows=3, cols=1, subplot_titles=("Box & Violin Plot", "Index vs Value Scatter", "Histogram"))
        fig.add_trace(go.Violin(x=self.df[column], box_visible=True, points='all', name="Distribution"), row=1, col=1)
        fig.add_trace(go.Scatter(y=self.df[column], mode='markers', name="Scatter"), row=2, col=1)
        fig.add_trace(go.Histogram(x=self.df[column], name="Histogram"), row=3, col=1)
        fig.update_layout(height=800, title_text=f"Univariate Analysis for {column}", showlegend=False)
        fig.show()

    def plot_relationship(self, col1, col2):
        if col1 not in self.df.columns or col2 not in self.df.columns: return
        type1 = 'Num' if self.df[col1].dtype in [np.number] else 'Cat'
        type2 = 'Num' if self.df[col2].dtype in [np.number] else 'Cat'
        if type1 == 'Num' and type2 == 'Num':
            fig = px.scatter(self.df, x=col1, y=col2, trendline="ols", title=f"Scatter Plot: {col1} vs {col2}")
        elif type1 == 'Cat' and type2 == 'Num':
            fig = px.box(self.df, x=col1, y=col2, points="all", title=f"Box Plot: {col2} by {col1}")
        elif type1 == 'Num' and type2 == 'Cat':
            fig = px.box(self.df, x=col2, y=col1, points="all", title=f"Box Plot: {col1} by {col2}")
        else:
            counts = self.df.groupby([col1, col2]).size().reset_index(name='Count')
            fig = px.bar(counts, x=col1, y='Count', color=col2, barmode='group', title=f"Grouped Bar Chart: {col1} & {col2}")
        fig.show()

    def _cramers_v(self, x, y):
        confusion_matrix = pd.crosstab(x, y)
        chi2 = stats.chi2_contingency(confusion_matrix)[0]
        n = confusion_matrix.sum().sum()
        r, k = confusion_matrix.shape
        if n == 0 or min(r-1, k-1) == 0: return 0
        return np.sqrt(chi2 / (n * min(r-1, k-1)))

    def _eta_squared(self, cat_col, num_col):
        groups = [group.dropna().values for name, group in self.df.groupby(cat_col)[num_col]]
        groups = [g for g in groups if len(g) > 0]
        if len(groups) < 2: return 0
        f_val, p_val = stats.f_oneway(*groups)
        df_between = len(groups) - 1
        df_within = len(self.df[num_col].dropna()) - len(groups)
        if (f_val * df_between + df_within) == 0: return 0
        return (f_val * df_between) / (f_val * df_between + df_within)

    def plot_all_associations_heatmap(self):
        if self.df is None: return
        cols = [c for c in self.df.columns if not c.endswith('_outlier')]
        n = len(cols)
        matrix = np.zeros((n, n))
        for i in range(n):
            for j in range(n):
                col1, col2 = cols[i], cols[j]
                t1 = 'Num' if self.df[col1].dtype in [np.number] else 'Cat'
                t2 = 'Num' if self.df[col2].dtype in [np.number] else 'Cat'
                if i == j: matrix[i][j] = 1.0
                elif t1 == 'Num' and t2 == 'Num': matrix[i][j] = self.df[col1].corr(self.df[col2])
                elif t1 == 'Cat' and t2 == 'Cat': matrix[i][j] = self._cramers_v(self.df[col1], self.df[col2])
                elif t1 == 'Cat' and t2 == 'Num': matrix[i][j] = np.sqrt(self._eta_squared(col1, col2))
                elif t1 == 'Num' and t2 == 'Cat': matrix[i][j] = np.sqrt(self._eta_squared(col2, col1))
        matrix = np.nan_to_num(matrix)
        fig = px.imshow(matrix, x=cols, y=cols, text_auto=".2f",
                        title="Unified Association Heatmap (Pearson's r / Cramér's V / Eta)",
                        color_continuous_scale='RdBu_r', zmin=-1, zmax=1)
        fig.show()

In [9]:
# 1. Class එක Initialize කිරීම
inspector = DataInspector()

# 2. Upload Data (මෙතනදී ඔයා ළඟ තියෙන CSV එකක් upload කරන්න)
print("Please upload your dataset (e.g., titanic.csv):")
inspector.upload_data()

# 3. Data Summary බැලීම
inspector.data_summary()

# 4. Missing Values Impute කිරීම
inspector.handle_missing_values(strategy='median')

# 5. Duplicates ඉවත් කිරීම
inspector.remove_duplicates()

# 6. Normalized සහ Encoded දත්ත එකතු කර ප්‍රදර්ශනය කිරීම
print("\n--- Normalized Combined Dataset Preview ---")
combined_df = inspector.merge_datasets(num_strategy='minmax', cat_strategy='onehot')
display(combined_df.head())

# 7. Unified Heatmap එක ඇඳීම
print("\n--- Generating Unified Heatmap ---")
inspector.plot_all_associations_heatmap()

Please upload your dataset (e.g., titanic.csv):


Saving Titanic-Dataset.csv to Titanic-Dataset (1).csv
Successfully loaded Titanic-Dataset (1).csv
Column 'Ticket' force-converted to Numeric.
--- DATASET SUMMARY ---
Rows: 891 | Columns: 12

Numerical Columns (8): ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare']
Categorical Columns (4): ['Name', 'Sex', 'Cabin', 'Embarked']

--- FIRST 20 ROWS PREVIEW ---


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,NaN,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,NaN,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,NaN,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803.0,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450.0,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877.0,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463.0,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909.0,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742.0,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736.0,30.0708,NaN,C


Missing values imputed using 'median' strategy.
Removed 0 duplicate rows.

--- Normalized Combined Dataset Preview ---


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Ticket,Fare,"Name_Abbott, Mr. Rossmore Edward","Name_Abbott, Mrs. Stanton (Rosa Hunt)",...,Cabin_F G63,Cabin_F G73,Cabin_F2,Cabin_F33,Cabin_F38,Cabin_F4,Cabin_G6,Cabin_T,Embarked_Q,Embarked_S
0,0.000000,0.0,1.0,0.271174,0.125,0.0,0.075946,0.014151,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,0.001124,1.0,0.0,0.472229,0.125,0.0,0.075946,0.139136,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.002247,1.0,1.0,0.321438,0.000,0.0,0.075946,0.015469,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.003371,1.0,0.0,0.434531,0.125,0.0,0.036480,0.103644,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.004494,0.0,1.0,0.434531,0.000,0.0,0.120221,0.015713,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0



--- Generating Unified Heatmap ---


In [10]:
# Univariate Subplots පරීක්ෂා කිරීම (Age column එක සඳහා)
inspector.plot_univariate_subplots('Age')

# Smart Relationship Chart එක පරීක්ෂා කිරීම (Sex සහ Survived අතර)
inspector.plot_relationship('Sex', 'Survived')